# Qwen3-8B story generation on Colab
Generate 14 emotions × 20 topics × 4 stories = **1,120 stories**, all generated by Qwen. No API key or Ollama needed. Use a **fresh GPU runtime**, separate from the extraction notebook, to avoid loading two models.

Run sections 1–6 for a one-combination benchmark. Inspect the four stories and timing, then run section 7 for the rest. Outputs are external JSONL files compatible with the extraction notebook. No stories are embedded here.

Drive is **off by default**, following your authentication issue. Download the checkpoint ZIP regularly: `/content` is temporary. To resume on a new runtime, restore that ZIP in section 3. If Drive authentication works later, use `USE_DRIVE=True`.

The prompts request indirect emotional portrayal and 100–160 words. Automated checks cover formatting, length, listed emotion terms, and exact duplicates; human review must establish intended emotion and detect near duplicates. All labels remain pending review. Generation and extraction use 4-bit NF4; record this adaptation to the paper.


## 1. Install
Run once in a fresh runtime. Restart the session if Colab requests it; continue at section 2. Dependencies stay on Transformers 4.x for compatibility with your extraction notebook. Unrelated preinstalled Gradio/Diffusers packages may report conflicts; this notebook does not import them.

In [1]:
%pip -q install "transformers>=4.51.0,<5" "accelerate>=1.6,<2" "bitsandbytes>=0.45,<1" safetensors tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Settings and topics
Keep these settings fixed when resuming. Topic IDs 0–13 are train, 14–16 validation, and 17–19 test. These are proposed project splits, not the paper’s exact splits.

In [2]:
import json, re, os, time, hashlib, shutil, zipfile, platform
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import importlib.metadata as metadata
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from huggingface_hub import HfApi

assert torch.cuda.is_available(), "Enable a GPU runtime first."
print(torch.cuda.get_device_name(0))
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/dsc180a_qwen_generation')
else:
    ROOT = Path('/content/dsc180a_qwen_generation')
ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ID = 'Qwen/Qwen3-8B'
MODEL_REVISION = 'main'  # resolved to a commit once and reused on resume
EMOTIONS = ['desperate','nervous','surprised','calm','angry','loving',
            'sad','afraid','inspired','happy','guilty','proud','frustrated','hopeful']
TOPICS = ['A neighbor starts a renovation project', 'An adult child moves back in with their parents', 'A character prepares a meal for unexpected guests', 'A character returns to their hometown after several years', 'A character begins their first day at a new job', 'A character searches for a missing personal item', 'A character receives a letter from someone they have not heard from in years', 'A character helps organize a community event', 'A character waits for an important phone call', 'A character takes care of a friend’s pet for the weekend', 'A character learns that their usual route to work is closed', 'A character sorts through belongings before moving', 'A character attends a family gathering', 'A character works with a stranger to complete a shared task', 'A character prepares for a public performance', 'A character discovers an old photograph', 'A character lends a possession to someone else', 'A character visits a place they have never been before', 'A character repairs a broken household object', 'A character encounters an old acquaintance at a grocery store']
STORIES_PER_PAIR = 4
BASE_SEED = 180
MAX_ATTEMPTS = 5  # attempts per incomplete pair per invocation
GENERATION = dict(max_new_tokens=1400, do_sample=True, temperature=0.7,
                  top_p=0.8, top_k=20, repetition_penalty=1.05)
TERMS = {
 'desperate':['desperate','desperation'],
 'nervous':['nervous','nervousness','anxious','anxiety','jittery'],
 'surprised':['surprised','surprise','astonished','shocked','amazed'],
 'calm':['calm','calmness','serene','peaceful','relaxed'],
 'angry':['angry','anger','furious','enraged','irate'],
 'loving':['loving','love','affection','adoration','devoted'],
 'sad':['sad','sadness','sorrow','sorrowful','unhappy'],
 'afraid':['afraid','fear','fearful','scared','frightened','terrified'],
 'inspired':['inspired','inspiration','inspiring','motivated','invigorated'],
 'happy':['happy','happiness','joy','joyful','delighted','cheerful'],
 'guilty':['guilty','guilt','remorse','remorseful','ashamed'],
 'proud':['proud','pride','prideful'],
 'frustrated':['frustrated','frustration','exasperated'],
 'hopeful':['hopeful','hope','optimistic','optimism'],
}
def split_for(t): return 'train' if t < 14 else 'validation' if t < 17 else 'test'
def normalize(s): return ' '.join(s.casefold().split())
DATA = ROOT / 'emotion_stories_qwen_full1120.jsonl'
MANIFEST = ROOT / 'manifest.json'
RAW = ROOT / 'attempts.jsonl'


Tesla T4


## 3. Optional restoration
For a **new runtime**, set `RESTORE_ZIP=True` and upload the previous checkpoint ZIP. On the same runtime, keep it false. Restore before loading the model. The ZIP includes settings and attempt history as well as accepted stories.

In [3]:
RESTORE_ZIP = False
if RESTORE_ZIP:
    from google.colab import files
    uploads = files.upload()
    if len(uploads) != 1:
        raise ValueError('Select one checkpoint ZIP.')
    name, blob = next(iter(uploads.items()))
    import io
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        for info in archive.infolist():
            if info.filename not in {'manifest.json', DATA.name, 'attempts.jsonl', 'summary.json'}:
                raise ValueError(f'Unexpected ZIP entry: {info.filename}')
            target = ROOT / info.filename
            if target.exists() and target.read_bytes() != archive.read(info):
                raise ValueError(f'Restore would overwrite a different file: {target}')
        archive.extractall(ROOT)
    print('Restored checkpoint.')


## 4. Pin and load the model
4-bit NF4 with float16 computation is used on the GPU. Thinking is disabled in the chat template. Model download can take several minutes. KV caching is enabled for generation; extraction uses different cache settings.

In [4]:
PROMPT_VERSION = 'indirect-emotion-four-stories-v1'
config = dict(model=MODEL_ID, emotions=EMOTIONS, topics=TOPICS,
              stories_per_pair=STORIES_PER_PAIR, base_seed=BASE_SEED,
              generation=GENERATION, terms=TERMS, prompt_version=PROMPT_VERSION,
              word_range=[100,160], thinking=False, quantization='NF4-double-fp16',
              topic_splits=[split_for(t) for t in range(len(TOPICS))])
if MANIFEST.exists():
    manifest = json.loads(MANIFEST.read_text())
    if manifest['config'] != config:
        raise ValueError('Configuration changed. Restore the original settings or use a new ROOT.')
    revision = manifest['model_revision']
else:
    if DATA.exists() or RAW.exists():
        raise ValueError('Output exists without manifest. Restore its manifest or use a new ROOT.')
    revision = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha
    manifest = dict(config=config, model_revision=revision,
                    created_utc=datetime.now(timezone.utc).isoformat(), sessions=[])

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=revision, device_map={'':0}, low_cpu_mem_usage=True,
    torch_dtype=torch.float16, attn_implementation='sdpa',
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,
        bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16))
model.eval()
model.config.use_cache = True
manifest['sessions'].append(dict(time=datetime.now(timezone.utc).isoformat(),
    gpu=torch.cuda.get_device_name(0), python=platform.python_version(),
    versions={p:metadata.version(p) for p in ['torch','transformers','accelerate','bitsandbytes','huggingface_hub']}))
MANIFEST.write_text(json.dumps(manifest, indent=2))
print('Model revision:', revision)
print('Footprint GiB:', round(model.get_memory_footprint()/2**30,2))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

... (7 outputs left out)

model-00002-of-00005.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model revision: b968826d9c46dd6066d109eabc6255188de91218
Footprint GiB: 5.55


## 5. Generation, checks, and checkpoint functions
Each request asks for four stories. A pair is saved only if all four pass the checks. Failed responses are logged and retried with a new seed. Token-limit truncation is rejected. Interrupting a request preserves previously saved pairs. Exact duplicates across all accepted stories are rejected; synonyms outside the list and semantic quality need human review.

In [8]:
def make_prompt(emotion, topic):
    return f"""Write 4 different stories based on the following premise.
Topic: {topic}
The story should follow a character who is feeling {emotion}.
Format exactly four stories like so:
<NEW STORY>
[story 1]
<NEW STORY>
[story 2]
<NEW STORY>
[story 3]
<NEW STORY>
[story 4]
The paragraphs should each be a fresh start, with no continuity. Try to make them
diverse and not use the same turns of phrase. Across the different stories,
use a mix of third-person narration and first-person narration.
IMPORTANT: You must NEVER use the word '{emotion}' or any direct synonyms of it in
the stories. Instead, convey the emotion ONLY through:
- The character‘s actions and behaviors
- Physical sensations and body language
- Dialogue and tone of voice
- Thoughts and internal reactions
- Situational context and environmental descriptions
The emotion should be clearly conveyed to the reader through these indirect means,
but never explicitly named."""

def read_jsonl(path):
    if not path.exists(): return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

def atomic_jsonl(path, rows):
    temp = path.with_suffix('.tmp')
    with temp.open('w', encoding='utf-8') as f:
        for row in rows: f.write(json.dumps(row, ensure_ascii=False)+'\n')
        f.flush()
        os.fsync(f.fileno())
    os.replace(temp, path)

def load_rows():
    rows = read_jsonl(DATA)
    ids, texts = set(), set()
    groups = Counter()
    for r in rows:
        e,t = r['emotion'],r['topic_id']
        assert e in EMOTIONS and isinstance(t,int) and 0 <= t < len(TOPICS)
        assert r['topic'] == TOPICS[t] and r['split'] == split_for(t)
        assert r['generator_model'] == MODEL_ID and r['model_revision'] == revision
        assert r['id'] == f'qwen3-8b-{e}-{t:02d}-{r["story_index"]}'
        assert r['story_index'] in range(4)
        assert r['id'] not in ids and normalize(r['text']) not in texts
        ids.add(r['id']); texts.add(normalize(r['text'])); groups[e,t]+=1
    assert all(n == 4 for n in groups.values()), 'Checkpoint contains an incomplete pair.'
    return rows

def parse_stories(raw, emotion, existing):
    parts = raw.split('<NEW STORY>')
    if len(parts)!=5 or parts[0].strip():
        return [], ['Expected exactly four <NEW STORY> markers, no preamble.']
    stories=[p.strip() for p in parts[1:]]
    issues=[]; seen=set(existing)
    for i,s in enumerate(stories):
        if '\n' in s: issues.append(f'{i}: multiple lines')
        hits=[w for w in TERMS[emotion] if re.search(r'\b'+re.escape(w)+r'\b',s,re.I)]
        if hits: issues.append(f'{i}: prohibited terms {hits}')
        if normalize(s) in seen: issues.append(f'{i}: exact duplicate')
        seen.add(normalize(s))
    return stories,issues

def export_checkpoint(download=False):
    archive = Path('/content/qwen_story_checkpoint.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for path in [MANIFEST,DATA,RAW,ROOT/'summary.json']:
            if path.exists(): z.write(path,path.name)
    if download:
        from google.colab import files
        files.download(str(archive))
    return archive

rows = load_rows()

def generate_pair(emotion, topic_id):
    global rows
    # Re-read disk so rerunning a cell cannot silently re-add completed pairs.
    rows=load_rows()
    if any(r['emotion']==emotion and r['topic_id']==topic_id for r in rows): return True
    previous=[r for r in read_jsonl(RAW) if r['emotion']==emotion and r['topic_id']==topic_id]
    offset=max([r['attempt'] for r in previous],default=-1)+1
    prompt=make_prompt(emotion,TOPICS[topic_id])
    for attempt in range(offset,offset+MAX_ATTEMPTS):
        seed=BASE_SEED + EMOTIONS.index(emotion)*100000 + topic_id*1000 + attempt
        set_seed(seed)
        chat=tokenizer.apply_chat_template([{'role':'user','content':prompt}],
            tokenize=False,add_generation_prompt=True,enable_thinking=False)
        inputs=tokenizer(chat,return_tensors='pt',add_special_tokens=False).to('cuda')
        print(f'{emotion} / topic {topic_id} / attempt {attempt}: generating...',flush=True)
        torch.cuda.synchronize(); start=time.monotonic()
        with torch.inference_mode():
            output=model.generate(**inputs,**GENERATION,use_cache=True,
                                  pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize(); elapsed=time.monotonic()-start
        new=output[0,inputs['input_ids'].shape[1]:]
        raw=tokenizer.decode(new,skip_special_tokens=True)
        stories,issues=parse_stories(raw,emotion,{normalize(r['text']) for r in rows})
        eos=model.generation_config.eos_token_id
        eos=set(eos if isinstance(eos,list) else [eos])
        if len(new)>=GENERATION['max_new_tokens'] and int(new[-1]) not in eos:
            issues.append('Output reached token cap without EOS.')
        log=dict(emotion=emotion,topic_id=topic_id,attempt=attempt,seed=seed,
                 seconds=elapsed,tokens=len(new),issues=issues,text=raw,prompt=prompt)
        with RAW.open('a',encoding='utf-8') as f:
            f.write(json.dumps(log,ensure_ascii=False)+'\n'); f.flush(); os.fsync(f.fileno())
        print(f'{elapsed:.1f}s, {len(new)/elapsed:.1f} tokens/s; issues: {issues}',flush=True)
        if issues: continue
        accepted=[dict(id=f'qwen3-8b-{emotion}-{topic_id:02d}-{i}',emotion=emotion,
            topic_id=topic_id,topic=TOPICS[topic_id],story_index=i,split=split_for(topic_id),
            text=s,word_count=len(s.split()),generator='Qwen',generator_model=MODEL_ID,
            model_revision=revision,seed=seed,attempt=attempt,flags=[],
            review_status='pending_human_review',dataset_version='qwen-full1120-v1')
            for i,s in enumerate(stories)]
        atomic_jsonl(DATA,rows+accepted)
        rows=rows+accepted
        export_checkpoint()
        return True
    print('Pair unresolved; logged for review. Rerunning retries with new seeds.')
    return False


## 6. Benchmark one real combination
Inspect the stories before starting the full run. The timing is one request, not an estimate based on CPU throughput. Retries will add time. This accepted pair counts toward the full dataset.

In [9]:
generate_pair(EMOTIONS[0],0)
for r in rows:
    if r['emotion']==EMOTIONS[0] and r['topic_id']==0:
        print('\n',r['id'],'\n',r['text'])
accepted_logs=[r for r in read_jsonl(RAW) if not r['issues']]
if accepted_logs:
    seconds=accepted_logs[-1]['seconds']
    remaining=280-len(rows)//4
    print(f'Rough remaining time without retries: {remaining*seconds/3600:.2f} hours.')
export_checkpoint(download=True)


desperate / topic 0 / attempt 5: generating...
47.0s, 12.3 tokens/s; issues: []

 qwen3-8b-desperate-00-0 
 The morning sun filtered through the curtains as Clara sat on the edge of her bed, staring at the ceiling. Her hands trembled slightly, and she kept checking her phone, though there were no new messages. The sound of a hammer echoed from across the street, where Mrs. Delaney had begun tearing down the old porch. It was louder than it should have been, and it made Clara’s chest feel tight. She stood up abruptly, knocking over a glass of water. The mess didn’t matter. What mattered was that the noise was getting worse, and the house next door was changing—just like everything else in her life felt like it was slipping away.

 qwen3-8b-desperate-00-1 
 I’ve always liked the way the old house looked. Its crooked roof, the peeling paint, the way the windows creaked when the wind blew. But now, every time I walk past the front steps, I see the scaffolding and the men in hard hats. My f

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

PosixPath('/content/qwen_story_checkpoint.zip')

## 7. Generate remaining pairs
Run after reviewing the benchmark. Progress prints before every request. Automatically downloads a checkpoint every 20 newly accepted pairs and at the end; your browser may ask to allow downloads. If interrupted, run section 8 to download immediately. On the same runtime, rerun this cell to resume. On a new runtime, restore the latest ZIP in section 3 first. Failed pairs remain missing and can be retried by rerunning.

In [10]:
from tqdm.auto import tqdm
new_pairs=0
try:
    for topic_id in tqdm(range(len(TOPICS)),desc='Topics'):
        for emotion in EMOTIONS:
            before=len(rows)
            generate_pair(emotion,topic_id)
            if len(rows)>before:
                new_pairs+=1
                if new_pairs%20==0: export_checkpoint(download=True)
finally:
    export_checkpoint(download=True)


Topics:   0%|          | 0/20 [00:00<?, ?it/s]

nervous / topic 0 / attempt 0: generating...
46.1s, 12.2 tokens/s; issues: []
surprised / topic 0 / attempt 0: generating...
48.3s, 12.0 tokens/s; issues: []
calm / topic 0 / attempt 0: generating...
44.0s, 12.2 tokens/s; issues: []
angry / topic 0 / attempt 0: generating...
47.0s, 12.1 tokens/s; issues: []
loving / topic 0 / attempt 0: generating...
91.8s, 12.5 tokens/s; issues: ['0: multiple lines', '1: multiple lines', '2: multiple lines', '3: multiple lines']
loving / topic 0 / attempt 1: generating...
42.5s, 12.4 tokens/s; issues: []
sad / topic 0 / attempt 0: generating...
41.9s, 12.4 tokens/s; issues: []
afraid / topic 0 / attempt 0: generating...
41.9s, 12.5 tokens/s; issues: []
inspired / topic 0 / attempt 0: generating...
33.7s, 12.5 tokens/s; issues: []
happy / topic 0 / attempt 0: generating...
37.7s, 12.4 tokens/s; issues: []
guilty / topic 0 / attempt 0: generating...
32.9s, 12.6 tokens/s; issues: []
proud / topic 0 / attempt 0: generating...
41.5s, 12.4 tokens/s; issues:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

afraid / topic 1 / attempt 0: generating...
44.4s, 12.0 tokens/s; issues: []
inspired / topic 1 / attempt 0: generating...
42.1s, 12.0 tokens/s; issues: []
happy / topic 1 / attempt 0: generating...
47.8s, 11.8 tokens/s; issues: []
guilty / topic 1 / attempt 0: generating...
44.5s, 12.0 tokens/s; issues: []
proud / topic 1 / attempt 0: generating...
43.1s, 12.3 tokens/s; issues: []
frustrated / topic 1 / attempt 0: generating...
42.4s, 12.3 tokens/s; issues: []
hopeful / topic 1 / attempt 0: generating...
51.4s, 12.1 tokens/s; issues: []
desperate / topic 2 / attempt 0: generating...
69.1s, 12.4 tokens/s; issues: ['0: multiple lines', '1: multiple lines', '2: multiple lines', '3: multiple lines']
desperate / topic 2 / attempt 1: generating...
42.6s, 12.2 tokens/s; issues: []
nervous / topic 2 / attempt 0: generating...
49.6s, 12.4 tokens/s; issues: []
surprised / topic 2 / attempt 0: generating...
52.1s, 12.4 tokens/s; issues: []
calm / topic 2 / attempt 0: generating...
47.7s, 12.3 to

<IPython.core.display.Javascript object>

... (33 outputs left out)

<IPython.core.display.Javascript object>

happy / topic 18 / attempt 0: generating...
45.7s, 13.0 tokens/s; issues: []
guilty / topic 18 / attempt 0: generating...
43.9s, 13.3 tokens/s; issues: []
proud / topic 18 / attempt 0: generating...
51.6s, 13.3 tokens/s; issues: ['0: multiple lines', '1: multiple lines', '2: multiple lines', '3: multiple lines']
proud / topic 18 / attempt 1: generating...
43.2s, 13.0 tokens/s; issues: []
frustrated / topic 18 / attempt 0: generating...
47.9s, 13.1 tokens/s; issues: []
hopeful / topic 18 / attempt 0: generating...
47.8s, 13.1 tokens/s; issues: []
desperate / topic 19 / attempt 0: generating...
38.7s, 12.8 tokens/s; issues: []
nervous / topic 19 / attempt 0: generating...
40.0s, 12.8 tokens/s; issues: []
surprised / topic 19 / attempt 0: generating...
38.5s, 12.9 tokens/s; issues: []
calm / topic 19 / attempt 0: generating...
40.5s, 13.0 tokens/s; issues: []
angry / topic 19 / attempt 0: generating...
45.2s, 13.1 tokens/s; issues: []
loving / topic 19 / attempt 0: generating...
48.8s, 12

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Coverage check and download
Missing pairs are reported explicitly. A partial file still loads in extraction, so confirm **1,120** before treating it as the full dataset. Automated acceptance does not verify emotional correctness. Review samples across all labels and topics before experiments. The ZIP’s JSONL can be uploaded directly to section 3 of your extraction notebook. Set that notebook’s model revision to the commit printed in section 4 for consistent model weights.

In [11]:
rows=load_rows()
counts=Counter((r['emotion'],r['topic_id']) for r in rows)
missing=[(e,t) for t in range(len(TOPICS)) for e in EMOTIONS if counts[e,t]!=4]
summary=dict(stories=len(rows),expected=1120,complete=not missing,
             split_counts=dict(Counter(r['split'] for r in rows)),missing_pairs=missing)
(ROOT/'summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('JSONL:',DATA)
export_checkpoint(download=True)


{
  "stories": 1120,
  "expected": 1120,
  "complete": true,
  "split_counts": {
    "train": 784,
    "validation": 168,
    "test": 168
  },
  "missing_pairs": []
}
JSONL: /content/dsc180a_qwen_generation/emotion_stories_qwen_full1120.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

PosixPath('/content/qwen_story_checkpoint.zip')

Sources: [Qwen3-8B model card](https://huggingface.co/Qwen/Qwen3-8B) (chat template, non-thinking generation settings); [Transformers bitsandbytes](https://huggingface.co/docs/transformers/quantization/bitsandbytes) (NF4 loading).

Validation: notebook code is syntax-checked and parsing/checkpoint helpers are exercised locally. Full model loading and GPU generation require a Colab run and have not been executed here. Seeds and pinned weights aid reproducibility but do not guarantee identical outputs across hardware/software versions.